In [9]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Check_Data").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 1. Đọc file dữ liệu
df = spark.read.csv("hdfs://localhost:9000/user/data/hotel_bookings.csv", header=True, inferSchema=True)

# 2. In ra kích thước
print("Số lượng mẫu tin (Rows):", df.count())
print("Số lượng thuộc tính (Columns):", len(df.columns))

Số lượng mẫu tin (Rows): 119390
Số lượng thuộc tính (Columns): 32


In [ ]:
!pip3 install numpy

In [7]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

spark = SparkSession.builder.appName("Preprocessing").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.csv("hdfs://localhost:9000/user/data/hotel_bookings.csv", header=True, inferSchema=True)
df = df.withColumn("children", df["children"].cast("int"))
df = df.withColumn("adr", df["adr"].cast("double"))

# 1. Xử lý missing value
df_cleaned = df.na.fill({"children": 0, "country": "Unknown"})

# 2. Định nghĩa các nhóm biến
categorical_cols = ["hotel", "meal", "deposit_type", "customer_type"]
numerical_cols = ["lead_time", "stays_in_weekend_nights", "stays_in_week_nights", "adults", "children", "adr"]

# 3. Xây dựng cấu phần của Pipeline
indexers = [StringIndexer(inputCol=col, outputCol=col+"_index", handleInvalid="keep") for col in categorical_cols]
encoders = [OneHotEncoder(inputCol=col+"_index", outputCol=col+"_vec") for col in categorical_cols]

assembler_inputs = [col+"_vec" for col in categorical_cols] + numerical_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)

# 4. Lắp ráp và chạy Pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])
pipeline_model = pipeline.fit(df_cleaned)
preprocessed_df = pipeline_model.transform(df_cleaned)

# 5. Chia tập Train/Test
train_df, test_df = preprocessed_df.randomSplit([0.8, 0.2], seed=42)

print("Cấu trúc dữ liệu sau khi tiền xử lý (Mẫu tin đầu tiên):")
preprocessed_df.select("features", "is_canceled").show(1, truncate=False)

print(f"Số lượng mẫu tin tập huấn luyện (Train): {train_df.count()}")
print(f"Số lượng mẫu tin tập kiểm thử (Test): {test_df.count()}")

Cấu trúc dữ liệu sau khi tiền xử lý (Mẫu tin đầu tiên):
+-------------------------------------------------------------------------------------------------------------------------------------+-----------+
|features                                                                                                                             |is_canceled|
+-------------------------------------------------------------------------------------------------------------------------------------+-----------+
|(20,[1,2,7,10,14,17],[2.117834085406122,2.3879071058150347,3.039022426054744,2.311213973853459,3.2003564321780926,3.452675053266355])|0          |
+-------------------------------------------------------------------------------------------------------------------------------------+-----------+
only showing top 1 row

Số lượng mẫu tin tập huấn luyện (Train): 95673
Số lượng mẫu tin tập kiểm thử (Test): 23717


In [6]:
# kiểm tra kết quả tiền xử lý
preprocessed_df.select("hotel", "hotel_index", "hotel_vec", "features").show(5, truncate=False)

+------------+-----------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hotel       |hotel_index|hotel_vec    |features                                                                                                                                                                          |
+------------+-----------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Resort Hotel|1.0        |(2,[1],[1.0])|(20,[1,2,7,10,14,17],[2.117834085406122,2.3879071058150347,3.039022426054744,2.311213973853459,3.2003564321780926,3.452675053266355])                                             |
|Resort Hotel|1.0        |(2,[1],[1.0])|(20,[1,2,7,10,14,17],[2.117834085406122,2.3879071058150347,3.039022426054744,2.3

In [8]:
# Gộp dữ liệu thành 1 phân vùng và xuất ra định dạng CSV có chứa tên cột
df_cleaned.coalesce(1).write.mode("overwrite").csv("Hotel_Cleaned_Data", header=True)

print("Đã xuất file thành công! Sẵn sàng cho team cá chạy SQL.")

Đã xuất file thành công! Sẵn sàng cho team cá chạy SQL.
